In [71]:
from pyconll.conllu import conllu
import pandas as pd

In [72]:
path = './dataset/ParlaMint-HU_2023-07-31.conllu'
metadata_path = './dataset/ParlaMint-HU_2023-07-31-meta-en.tsv'
file_to_process = conllu.load_from_file(path)

One issue is that the key is not consistent between sentences to be able to join on the data in the meta.tsv files. They rely on the first sentence in the speech being labelled with 'newdoc id' and there are different IDs for individual utterances, but in order to classify at the sentence level we must associate each utterance with the speaker/party/orientation and therefore the class label. We need to create a consistent join key for each sentence.

In [73]:
def create_join_keys(file):
    current_id = ''
    for sentence in file:
        if sentence.meta.get('newdoc id', None) is not None:
            current_id = sentence.meta['newdoc id']
            sentence.meta.update({'join key': current_id})
        else:
            sentence.meta.update({'join key': current_id})
    return file

In [74]:
def file_to_token_array_dataframe(file):
    sentence_array = []
    for sentence in file:
        token_array = []
        for token in sentence.tokens:
            token_array.append(token.form)
        sentence_array.append({'tokens': token_array, 'join_key': sentence.meta['join key']})
    return pd.DataFrame(sentence_array)


In [75]:
def process_file(file):
    file_with_join_keys = create_join_keys(file)
    return file_to_token_array_dataframe(file_with_join_keys)

In [76]:
def process_metadata(file_path):
    return pd.read_csv(file_path, sep='\t')


In [77]:
process_file(file_to_process)

,tokens,join_key
0,"[Szép, jó, napot, kívánok, !]",u2023-07-31-0
1,"[Tisztelt, Hölgyeim, és, Uraim, !]",u2023-07-31-0
2,"[Tisztelt, Képviselőtársaim, !]",u2023-07-31-0
3,"[Az, Országgyűlés, nyári, 3., rendkívüli, ülés...",u2023-07-31-0
4,"[Tájékoztatom, önöket, ,, hogy, az, ülés, veze...",u2023-07-31-0
...,...,...
312,"[Ennek, alapján, a, határozati, házszabály, 14...",u2023-07-31-18
313,"[Elfogadott, napirendi, javaslat, hiányában, a...",u2023-07-31-18
314,"[Tájékoztatom, önöket, arról, is, ,, hogy, az,...",u2023-07-31-18
315,"[Az, ülést, bezárom, .]",u2023-07-31-18


In [78]:
def get_metadata_dataframe(file_path):
    df = pd.read_csv(file_path, sep='\t')
    df = df.rename({'ID': 'join_key'}, axis='columns')
    return df[['join_key', 'Date', 'Speaker_role', 'Speaker_MP', 'Speaker_minister', 'Speaker_party', 'Party_orientation', 'Speaker_ID', 'Speaker_name']]

In [85]:
def create_joint_dataframe(text_dataframe, metadata_dataframe):
    return text_dataframe.merge(metadata_dataframe, on='join_key', how='outer')

In [86]:
meta = get_metadata_dataframe(metadata_path)
text = process_file(file_to_process)
create_joint_dataframe(text, meta)

,tokens,join_key,Date,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Party_orientation,Speaker_ID,Speaker_name
0,"[Szép, jó, napot, kívánok, !]",u2023-07-31-0,2023-07-31,Chairperson,MP,notMinister,KNDP-frakció,Right,LatorcaiJanos,"Latorcai, János"
1,"[Tisztelt, Hölgyeim, és, Uraim, !]",u2023-07-31-0,2023-07-31,Chairperson,MP,notMinister,KNDP-frakció,Right,LatorcaiJanos,"Latorcai, János"
2,"[Tisztelt, Képviselőtársaim, !]",u2023-07-31-0,2023-07-31,Chairperson,MP,notMinister,KNDP-frakció,Right,LatorcaiJanos,"Latorcai, János"
3,"[Az, Országgyűlés, nyári, 3., rendkívüli, ülés...",u2023-07-31-0,2023-07-31,Chairperson,MP,notMinister,KNDP-frakció,Right,LatorcaiJanos,"Latorcai, János"
4,"[Tájékoztatom, önöket, ,, hogy, az, ülés, veze...",u2023-07-31-0,2023-07-31,Chairperson,MP,notMinister,KNDP-frakció,Right,LatorcaiJanos,"Latorcai, János"
...,...,...,...,...,...,...,...,...,...,...
312,"[Egy, felelős, kormány, ,, amely, törődne, a, ...",u2023-07-31-9,2023-07-31,Regular,MP,notMinister,JOBBIK-frakció,Centre-right,BrennerKoloman,"Brenner, Koloman"
313,"[Éspedig, ez, az, elvándorlás, az, egyik, legn...",u2023-07-31-9,2023-07-31,Regular,MP,notMinister,JOBBIK-frakció,Centre-right,BrennerKoloman,"Brenner, Koloman"
314,"[Nem, véletlen, talán, ,, hogy, a, Fidesz, töb...",u2023-07-31-9,2023-07-31,Regular,MP,notMinister,JOBBIK-frakció,Centre-right,BrennerKoloman,"Brenner, Koloman"
315,"[Ezért, is, indítottuk, el, újra, a, bérfelzár...",u2023-07-31-9,2023-07-31,Regular,MP,notMinister,JOBBIK-frakció,Centre-right,BrennerKoloman,"Brenner, Koloman"
